<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/ClassNotebooks/10-supervised_classification/10-supervised_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10: Supervised Learning I, Classification

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- [scikit-learn user guide](https://scikit-learn.org/stable/supervised_learning.html), on supervised learning
- Palmer Penguins (Horst, Hill, and Gorman, 2020; CC0)
- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

*Last major revision: 2026-10-29*

Module 9 clustered data that had no answer key. Today the answer key exists, and everything changes: we can finally measure whether the model is any good.

## Key Concepts

- Distinguish features from labels, and classification from regression
- Explain why a model must be evaluated on data it has never seen, and perform a **train/test split**
- Carry out **k-nearest-neighbors** by hand, and explain why `k` and feature scaling both change the answer
- Compute the **Gini index** and **Gini gain** by hand to choose a decision-tree split
- Explain how a **random forest** combines bootstrap samples, random feature subsets, and a majority vote, and why it is steadier than one deep tree
- Read a **confusion matrix**, and compute accuracy, precision, recall, and F1
- Explain why accuracy alone is misleading on imbalanced data

---
## 1. Learning With an Answer Key
---

Last module a store asked "what kinds of customers do we have?" and no column held the answer. Today the question changes: *"which customers will respond to this campaign?"* and the store has a column recording who responded last time.

That column is a **label**.

**Definition.** In **supervised learning** each case comes with a known label. The **features** ($X$) are the inputs used to predict, and the **label** ($y$) is what we predict. A **model** is a function that maps $X$ to $y$; **training** is the automatic search for the version of that function which fits the known labels best.

There are two main flavors of supervised learning decided entirely by what kind of thing $y$ is:

- **Classification:** the label is a category and the possible categories are called classes.  *Which species is this penguin? Is this call fraudulent?*
- **Regression:** the label is a number. *How long is this penguin's flipper? What will this house sell for?*

In this Module we will explore classification. Module 11 addresses regression.

**A little vocabulary that will keep recurring.**

- **Parameters** are learned by the model during training. You do not choose them.
- **Hyperparameters** are settings *you* choose before training, like `k` in k-nearest-neighbors or `max_depth` in a decision tree. The model cannot learn them for you.
- **Prediction** (or inference) is applying the trained model to new cases.

In [ ]:
# RUN-TOGETHER
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

penguins = sns.load_dataset('penguins').dropna()
print(penguins.shape)
penguins.head(3)

---
## 2. The Train/Test Split
---

### The central problem of supervised learning:

>We want a model that works on data it has never seen. But we only have data we *have* seen. If we train a model on all of it and then measure how well it does on that same data, we learn nothing about the future: we have measured the model's memory, not its judgment.

### The Memorization Trap

Training on the entire dataset is like the teacher handing out the exact final exam questions and the answer key. A student who memorizes the exact answers to those specific questions has not learned how to solve new problems. When asked a slightly different question tomorrow, the student completely fails because they never learned the underlying concepts.

**Definition.** A **train/test split** holds part of the data back. The model is trained on the **training set** only, and evaluated on the **test set**, which it has never seen. The test score estimates performance on genuinely new data.

Note: **the test set must not influence training in any way.** Not the model, not the scaling, not the choice of `k`. The moment it does, your estimate is optimistic and you have no way to tell by how much.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_train_test_split.png?raw=true" width="720">

In [ ]:
# RUN-TOGETHER
from sklearn.model_selection import train_test_split

# Features: the four body measurements. Label: the species.
X = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = penguins['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=210, stratify=y)

print('training cases:', X_train.shape[0])
print('test cases    :', X_test.shape[0])
print()
print('species mix, training:'); print(y_train.value_counts(normalize=True).round(3))

`test_size=0.25` holds back a quarter. `random_state` fixes the shuffle so we all get the same split. `stratify=y` keeps the species proportions the same in both halves, which matters whenever one class is rare.

> **Discuss.** Suppose you tried 40 different models and reported the best test score. In what sense has the test set now influenced your choice, and is that score still an honest estimate of future performance? (Section 7.2 of Module 11 gives this problem its name and its fix.)

---
## 3. k-Nearest Neighbors (kNN)
---
### 3.1 The whole algorithm:

To predict the label of a new point $x$:

1. Compute the distance from $x$ to every point in the training data.
2. Keep the $k$ closest ones.
3. Predict the **majority label** among those $k$ neighbors.

That is the entire algorithm. There is no equation to solve and nothing is optimized; the "training" consists of storing the data. What makes it a *learning* algorithm is that we specify only the procedure, and the data supplies the answers.

The distance in step 1 is usually **Euclidean**:

$$d(a, b) \;=\; \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

which for two features is the ordinary straight-line distance you already know.

The same procedure serves both tasks. For **classification** the k neighbors vote (majority wins); for **regression** (Module 11) their values are averaged.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_knn_algorithm.png?raw=true" width="760">

#### **Activity 10.1 - k-nearest-neighbors by hand**

Six labeled training points, two features:

| point | $x_1$ | $x_2$ | label |
| --- | --- | --- | --- |
| P1 | 4 | 4 | **A** |
| P2 | 6 | 4 | **A** |
| P3 | 3 | 5 | **A** |
| P4 | 5 | 6 | **B** |
| P5 | 10 | 10 | **B** |
| P6 | 1 | 9 | **B** |

A new point arrives: $Q = (5, 5)$. What is its label?

**Step 1. Compute the distance from Q to each point.** Use $d = \sqrt{(x_1 - 5)^2 + (x_2 - 5)^2}$. Leave answers as square roots or round to two decimals.

| point | $(x_1 - 5)^2$ | $(x_2 - 5)^2$ | $d$ | label |
| --- | --- | --- | --- | --- |
| P1 |  |  |  | A |
| P2 |  |  |  | A |
| P3 |  |  |  | A |
| P4 |  |  |  | B |
| P5 |  |  |  | B |
| P6 |  |  |  | B |

**Step 2. Sort by distance.** List the points nearest to furthest:

____ , ____ , ____ , ____ , ____ , ____

**Step 3. Vote.**

- $k = 1$: the nearest neighbor is ______, so predict ______.
- $k = 3$: the three nearest are ______, ______, ______ with labels ______, so predict ______.
- $k = 5$: predict ______.

**Questions.**

**A.** Your answers for $k = 1$ and $k = 3$ **disagree**. Explain in one sentence what the data looks like near $Q$ that causes this.

**B.** Which answer do you trust more, and why? Is there a fact about the data that would change your mind?

**C.** `k` is a hyperparameter: nothing in the data tells you its value. What would happen if you set $k = 6$ here? What does that suggest about very large `k`?

**D.** Sketch the six points and $Q$ on graph paper. Does the picture agree with your arithmetic?

#### **Class Example 10.1 - The same six points, in code**

In [ ]:
# RUN-TOGETHER
from sklearn.neighbors import KNeighborsClassifier

X_toy = np.array([[4,4],[6,4],[3,5],[5,6],[10,10],[1,9]])
y_toy = np.array(['A','A','A','B','B','B'])
Q = np.array([[5,5]])

for k in (1, 3, 5):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_toy, y_toy)
    print(f'k = {k}:  predicts {knn.predict(Q)[0]}')

print()
dist, idx = KNeighborsClassifier(n_neighbors=6).fit(X_toy, y_toy).kneighbors(Q)
print('distances, nearest first:', dist.round(3).ravel())
print('their labels            :', y_toy[idx].ravel())

Check the distances against your table. The prediction really does flip from **B** to **A** as `k` goes from 1 to 3, and no amount of cleverness resolves which is correct: it is a choice about how much to trust the single closest point.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_knn_by_hand.png?raw=true" width="520">

*The rings sit at d = 1, $\sqrt{2}$, and 2. P4 alone is inside the first ring, which is why k=1 says B; widening to three neighbors pulls in two A's.*

This flip is not a quirk of our six points. Panel **A** below shows the same thing happening between k=3 and k=5, and panel **B** shows a related hazard: a query point far outside the data still receives a confident prediction, because *some* points are always nearest.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_knn_caveats.png?raw=true" width="760">

**The three limitations of k-NN:**

1. **You must choose `k`, and `k` changes the answer.** Activity 10.1 flipped from B to A; panel A above flips between k=3 and k=5. Nothing in the data tells you the right value.
2. **Features must be on comparable scales, or one column silently decides everything.** 
3. **It always gives a prediction, however far the new point sits from the data.** Panel B above: a query out in empty space still has a nearest neighbor, so it still gets a confident label. k-NN has no way to say "I have never seen anything like this."


### 3.2 Why scaling is not optional

k-NN measures distance, and distance is measured in whatever units the columns happen to use. Look at what our penguin features actually range over.

In [ ]:
# RUN-TOGETHER
print(X_train.describe().loc[['min', 'max']].round(1))

`body_mass_g` spans thousands; `bill_depth_mm` spans about ten. In the Euclidean formula those are squared and summed together, so a 200-gram difference in mass swamps a 5-mm difference in bill depth entirely. **Without scaling, k-NN on this data is very nearly k-NN on body mass alone.**

That is not a hypothetical. Here are three real penguins from this dataset.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_knn_scaling.png?raw=true" width="880">

*Q is an **Adelie**. In raw units its nearest neighbor is A, a **Gentoo**, purely because the two happen to weigh exactly the same (4700 g). Standardize, and the nearest neighbor becomes B, another **Adelie** with a similar bill depth. Nothing about the penguins changed; only the ruler did, and with it the predicted species.*

The repair is from Module 9. Standardize every column to mean 0 and standard deviation 1. 

**Only scale the TRAINING data!**

In [ ]:
# RUN-TOGETHER
from sklearn.preprocessing import StandardScaler

# Fit the scaler on the TRAINING data only, then apply it to both halves.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print('training means after scaling:', X_train_s.mean(axis=0).round(3))
print('training sds after scaling  :', X_train_s.std(axis=0).round(3))

**Why fit the scaler on training data only?** Because the test set is supposed to be data we have never seen, and computing its mean requires seeing it. Fitting the scaler on everything leaks information from the test set into training, and the resulting score is optimistic. This mistake is extremely common and completely invisible in the output.

### 3.3 k-NN on the penguins

In [ ]:
# RUN-TOGETHER
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
pred_knn = knn.predict(X_test_s)

print('test accuracy, scaled  :', round(knn.score(X_test_s, y_test), 3))

# For contrast: the same model on UNSCALED features.
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print('test accuracy, unscaled:', round(knn_raw.score(X_test, y_test), 3))

> **Discuss.** Both numbers come from the same algorithm, the same data, and the same `k`. Only the units changed. Which of the three limitations of k-NN does this illustrate, and what would you have concluded if you had only ever run the unscaled version?

One more view of `k`, this time as a picture of the whole decision surface rather than a single prediction. These are the penguins again, using just bill length and bill depth so the surface can be drawn.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_knn_boundary_vs_k.png?raw=true" width="960">

*At **k = 1** the boundary wraps around individual points, leaving small islands of one class stranded inside another. Training accuracy is a perfect 1.000, and that is the problem: the model has memorized every point, noise included. At **k = 15** the islands are gone but the genuine Adelie-Chinstrap boundary survives. At **k = 75** the surface is nearly straight and has begun slicing through real Chinstraps, and training accuracy has fallen to 0.949.*

---
## 4. Decision Trees
---
### 4.1 The idea

A decision tree asks a sequence of yes/no questions about the features, and each answer sends a case down a branch until it reaches a leaf carrying a prediction. It is the one algorithm in this course whose output a non-specialist can read directly.

The only real question is: **which question should the tree ask first?** A good split separates the classes; a bad one leaves them mixed. To choose automatically, we need to measure "mixed".

### 4.2 The Gini index

**Definition.** For a node containing $K$ classes, with $p_i$ the proportion of class $i$, the **Gini index** is

$$G \;=\; 1 - \sum_{i=1}^{K} p_i^{\,2}$$

It measures **impurity**. Read the extremes to get the feel of it:

- All one class: $p_1 = 1$, so $G = 1 - 1 = 0$. Perfectly pure.
- Two classes evenly split: $p_1 = p_2 = 0.5$, so $G = 1 - (0.25 + 0.25) = 0.5$. Maximally mixed, for two classes.

**What the number means.** Reach into a node, pull out a case at random, put it back, and pull out a second case. The chance that both draws come from class $i$ is $p_i \cdot p_i = p_i^2$, so $\sum p_i^2$ is the chance that the two draws **match**, and

$$G = 1 - \sum_{i=1}^{K} p_i^{\,2} = \text{the chance that two randomly drawn cases belong to } \textbf{different} \text{ classes.}$$

A pure node can never produce a disagreement, so its Gini is 0. An evenly mixed two-class node disagrees half the time, so its Gini is 0.5.

Here is how that chance grows as a node of 10 cases becomes more mixed. For two classes the formula simplifies to $G = 2p(1 - p)$, where $p$ is the proportion of either class.

| node (A, B) | 10, 0 | 9, 1 | 8, 2 | 7, 3 | 6, 4 | 5, 5 |
| --- | --- | --- | --- | --- | --- | --- |
| Gini | 0.00 | 0.18 | 0.32 | 0.42 | 0.48 | 0.50 |

Notice the shape. The first misfit to enter a pure node raises the Gini by 0.18, but moving from 6-4 to 5-5 raises it by only 0.02. In practice, this means Gini cares most about whether a child is **nearly pure**, which is exactly what a useful split produces.

A split is good when it produces children that are purer than the parent. Since a split makes two children, we compare the parent to the *weighted average* of the two.

**Definition.** The **Gini gain** of a split is

$$\Delta G \;=\; G_{\text{parent}} \;-\; \left( \frac{n_L}{n} G_L + \frac{n_R}{n} G_R \right)$$

where $n_L$ and $n_R$ are the numbers of cases sent left and right. **The tree tries every candidate split and keeps the one with the largest gain.**

Two candidate splits on the same twenty shapes. The left column shows what each threshold does to the mix of classes; the right shows the same points in feature space.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_decision_tree_ginigain.png?raw=true" width="820">

*Neither split is pure, but one leaves the children less mixed than the other. Gini gain is how we make "less mixed" a number.*

#### **Activity 10.2 - Choosing a split by hand**

Eight penguins, sorted by bill length. Two species: **A** (Adelie) and **G** (Gentoo).

| bill length (mm) | 35 | 37 | 39 | 41 | 43 | 45 | 47 | 49 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| species | A | A | A | G | G | A | G | G |

We will evaluate two candidate splits and pick the better one.

**Step 1. The parent node.** All eight penguins are in it.

- count of A = ______   count of G = ______
- $p_A$ = ______   $p_G$ = ______
- $G_{\text{parent}} = 1 - (p_A^2 + p_G^2)$ = ______

**Step 2. Split 1: bill length < 40.**

| | goes left (< 40) | goes right ($\geq$ 40) |
| --- | --- | --- |
| which penguins? |  |  |
| count A |  |  |
| count G |  |  |
| $n$ |  |  |
| Gini |  |  |

Weighted child Gini = $\frac{n_L}{8} G_L + \frac{n_R}{8} G_R$ = ______

Gini gain = ______

**Step 3. Split 2: bill length < 42.**

| | goes left (< 42) | goes right ($\geq$ 42) |
| --- | --- | --- |
| which penguins? |  |  |
| count A |  |  |
| count G |  |  |
| $n$ |  |  |
| Gini |  |  |

Weighted child Gini = ______   Gini gain = ______

**Step 4. Which split would the tree choose?** ______

**Questions.**

**A.** One of your two splits produced a child with Gini exactly 0. What does that mean about the penguins in it?

**B.** Split 2 puts the boundary only 2 mm away from Split 1, yet the gains differ substantially. Looking at the table, which single penguin is responsible?

**C.** The penguin at 45 mm is an A sitting among Gs. No threshold on bill length alone can fix that. Name two things the tree could do next, given that it is allowed to keep splitting.

**D.** Gini is 0 for a pure node and 0.5 for an evenly mixed two-class node. What would the Gini be for a node with three classes in equal proportion? Is it still capped at 0.5?

#### **Class Example 10.2 - The same eight penguins, in code**

In [ ]:
# RUN-TOGETHER
def gini(n_a, n_g):
    n = n_a + n_g
    if n == 0: return 0.0
    return 1 - ((n_a/n)**2 + (n_g/n)**2)

bill    = [35, 37, 39, 41, 43, 45, 47, 49]
species = ['A','A','A','G','G','A','G','G']

parent = gini(species.count('A'), species.count('G'))
print(f'parent Gini = {parent:.4f}\n')

for thr in (40, 42):
    L = [s for b, s in zip(bill, species) if b <  thr]
    R = [s for b, s in zip(bill, species) if b >= thr]
    gL, gR = gini(L.count('A'), L.count('G')), gini(R.count('A'), R.count('G'))
    weighted = len(L)/8*gL + len(R)/8*gR
    print(f'split at {thr}:  left {L} Gini={gL:.4f} | right {R} Gini={gR:.4f}')
    print(f'            weighted = {weighted:.4f}   GAIN = {parent - weighted:.4f}\n')

#### **Spam Filter Example, Part 1 - Growing two trees**

Activity 10.2 chose one split on one feature, and Question C asked what the tree could do next. A real tree does three more things:

1. It tries **every feature**, not just one.
2. It keeps splitting any child that is still mixed.
3. It stops when a node is pure, or when it reaches a limit you set, such as `max_depth`.

That is Activity 10.2's arithmetic repeated many times, which is exactly what scikit-learn's `DecisionTreeClassifier` does for us. Here we let it grow two trees on a tiny dataset, and concentrate on reading what it built.

**The data.** A tiny spam filter. Each email is described by two counts: how many exclamation marks it contains and how many links. *These eight emails are invented for this example.*

| email | exclamation marks | links | label |
| --- | --- | --- | --- |
| 1 | 0 | 0 | not spam |
| 2 | 1 | 6 | spam |
| 3 | 2 | 9 | spam |
| 4 | 3 | 1 | not spam |
| 5 | 5 | 3 | not spam |
| 6 | 6 | 4 | not spam |
| 7 | 8 | 2 | spam |
| 8 | 9 | 7 | spam |

We grow the same data twice: once allowed **one** question (`max_depth=1`) and once allowed **two** (`max_depth=2`).

In [ ]:
# RUN-TOGETHER
from sklearn.tree import DecisionTreeClassifier, export_text

emails = pd.DataFrame({
    'exclamation marks': [0, 1, 2, 3, 5, 6, 8, 9],
    'links':             [0, 6, 9, 1, 3, 4, 2, 7],
    'label': ['not spam', 'spam', 'spam', 'not spam', 'not spam', 'not spam', 'spam', 'spam']})
E = emails[['exclamation marks', 'links']]

spam_tree_a = DecisionTreeClassifier(max_depth=1, random_state=0).fit(E, emails['label'])
spam_tree_b = DecisionTreeClassifier(max_depth=2, random_state=0).fit(E, emails['label'])

print('Tree A (max_depth=1):')
print(export_text(spam_tree_a, feature_names=list(E.columns)))
print('Tree B (max_depth=2):')
print(export_text(spam_tree_b, feature_names=list(E.columns)))

**Reading the trees.** Each `|---` line is one branch, and indentation shows depth. Both trees open with the same question, **links $\leq$ 5**. Of the 14 thresholds the two features allow, that one had the largest Gini gain: the Activity 10.2 calculation, repeated 14 times. It sends emails 2, 3 and 8 (all spam) one way and the other five (4 not spam, 1 spam) the other.

> **Tree A:** if links $\leq$ 5 then **not spam**, otherwise **spam**.
>
> **Tree B:** if links > 5 then **spam**. Otherwise ask a second question: if exclamation marks $\leq$ 7 then **not spam**, otherwise **spam**.

Tree A stopped after one question, so its "not spam" leaf still contains email 7, which is spam. Tree B was allowed a second question, and *exclamation marks $\leq$ 7* separates those five emails perfectly.

**Using the trees.** Two new emails arrive. Run each one down both trees.

| new email | exclamation marks | links | Tree A | Tree B |
| --- | --- | --- | --- | --- |
| N1 | 8 | 3 | links $\leq$ 5, so **not spam** | links $\leq$ 5, then 8 > 7, so **spam** |
| N2 | 0 | 8 | links > 5, so **spam** | links > 5, so **spam** |

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_tree_by_hand.png?raw=true" width="720">

*Tree B drawn in feature space. The first question is the solid horizontal cut at links = 5; the second is the dashed vertical cut at 7 exclamation marks, which only exists below the first cut. Two cuts make three rectangles.*

**What to notice.**

- **Tree A and Tree B disagree about N1.** Deciding which tree is better needs emails that neither tree was grown from. Part 2 of this example, in Section 6, makes that comparison.
- **The regions are rectangles.** Every question a tree asks is a single threshold on a single feature, so every cut is a straight vertical or horizontal line. Tree B never looks at exclamation marks once an email has more than 5 links: N2 could be an ordinary department newsletter that happens to contain many links, and both trees call it spam without asking anything else.
- **Tree B classifies all eight emails it was grown from correctly.** With eight emails that is no achievement: a tree that may keep splitting can always do it, unless two emails with identical features carry different labels. Section 4.4 shows what that habit costs.

### 4.3 Growing a tree on the penguins

Trees need no scaling: a threshold on grams does not care that another column is in millimeters. This is a real practical advantage over k-NN.

In [ ]:
# RUN-TOGETHER
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
pred_tree = tree.predict(X_test)

print('test accuracy:', round(tree.score(X_test, y_test), 3))

fig, ax = plt.subplots(figsize=(13, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=tree.classes_,
          filled=True, rounded=True, fontsize=8)
plt.show()

Read the top box. It reports the feature and threshold chosen, the `gini` of the node, how many samples reached it, and the class counts. **That `gini` value is the quantity you computed by hand in Activity 10.2**, and the split shown is the one with the largest gain among all features and all thresholds.

The spam trees carved their feature space into rectangles, and so does every tree. Every split is a vertical or horizontal cut, so the regions are boxes, never curves or diagonals.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_tree_partition.png?raw=true" width="880">

*Compare this to the k-NN boundary above. Two algorithms, two very different vocabularies of shape.*

### 4.4 The tree's characteristic failure: overfitting

`max_depth=3` was a choice. What happens without one?

In [ ]:
# RUN-TOGETHER
for depth in [1, 2, 3, 5, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f'max_depth={str(depth):5}  train={t.score(X_train, y_train):.3f}  test={t.score(X_test, y_test):.3f}')

Watch the two columns separate. With no depth limit the tree drives **training** accuracy to 1.000 by growing branches until every training penguin sits in its own pure leaf. Test accuracy does not follow.

**Definition.** A model **overfits** when it learns detail specific to the training data that does not generalize. Its symptom is exactly what you see here: training score far above test score.

This is why the train/test split from Section 2 is not bureaucracy. Without it, the unlimited tree looks perfect.

**You have seen this failure before.** k-NN with $k = 1$ also scored a perfect 1.000 on its training penguins, by drawing islands around single points (the left panel of the boundary figure at the end of Section 3). An unlimited tree and a 1-nearest-neighbor model fail the same way: each has enough freedom to memorize the training data. `k` and `max_depth` are both dials on that freedom, turned in opposite directions: a **small** `k` and a **large** depth both mean more freedom, and more risk of overfitting.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_overfitting_depth.png?raw=true" width="760">

*Test accuracy peaks at depth 5 and then slips, while training accuracy climbs to a perfect 1.000. The widening shaded gap is the overfitting. Note how modest the penalty is here: the penguins are an easy, clean dataset. On messier data the test curve can fall off sharply, and the gap is your only warning.*

---
## 5. Random Forests: Many Trees, One Vote
---
### 5.1 The idea

Section 4.4 left the decision tree with a problem. Grown deep, it fits the training penguins perfectly and memorizes their noise. There is a second, quieter problem: a deep tree is **unstable**. Remove a few training penguins, or add a few, and the splits near the top can move, which redraws every box beneath them.

The fix is the same one a sensible person uses for an important decision: do not trust one opinionated expert, ask a panel and take the majority. But a panel only helps if its members make **different** mistakes. Five copies of the same expert vote five times for the same wrong answer.

A random forest manufactures that disagreement on purpose, using two sources of randomness.

**Definition.** A **bootstrap sample** of $n$ cases is $n$ cases drawn from the data **with replacement**. Some cases are picked twice or more, and some are not picked at all. Any one case is left out with probability $(1 - 1/n)^n$, which is close to $1/e \approx 0.37$ once $n$ is more than a few dozen, so each tree misses roughly a third of the training data.

**Definition.** A **random forest** grows many deep decision trees. Each tree is grown on its own bootstrap sample of the training data, and at every split it may choose only from a **random subset of the features**. To classify a new case, every tree votes, and the **majority wins**.

You have seen voting before: in k-NN the $k$ nearest neighbors voted. Here the voters are trees.

Two hyperparameters matter most:

- `n_estimators`: how many trees. More trees make the vote steadier and cost more time; adding trees does not by itself cause overfitting.
- `max_features`: how many features each split may consider. In scikit-learn the default for classification is `"sqrt"`, the square root of the number of features. With our four penguin measurements, each split chooses from a random **2** of the 4.

*Implementation note:* scikit-learn does not literally count votes. Each tree reports class proportions from its leaf, and the forest averages those across trees and picks the class with the highest average. A fully grown tree's leaves are almost always pure, so each tree's report is a single class, and the average becomes the vote share. Section 5.3 checks that the two agree on our data.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_forest_vs_tree.png?raw=true" width="960">

*Left and middle: two deep trees, one grown on all the training penguins and one on a bootstrap resample of them. Both are a staircase of boxes, and they draw the Adelie-Chinstrap border in visibly different places (look near a bill length of 45 mm). Right: a forest of 200 such trees voting. All three score 0.940 on the test set, so on these two features the forest has not bought accuracy. What it has bought is a boundary that depends less on which penguins happened to land in the training set. Notice also the bottom-left corner of the right panel, where there are no penguins at all: the forest still paints a confident answer there, just as k-NN did in panel B of the k-NN caveats figure.*

### 5.2 A forest on the penguins

Like a single tree, a forest needs **no scaling**: it is built entirely from thresholds. We use the same unscaled `X_train` as Section 4.

In [ ]:
# RUN-TOGETHER
from sklearn.ensemble import RandomForestClassifier

deep_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print(f'one deep tree :  train={deep_tree.score(X_train, y_train):.3f}  test={deep_tree.score(X_test, y_test):.3f}')
print(f'random forest :  train={rf.score(X_train, y_train):.3f}  test={rf.score(X_test, y_test):.3f}')

Both models fit the training penguins perfectly, which in Section 4.4 was the warning sign of overfitting. Yet the forest's test score holds up: averaging many overfit trees cancels much of the noise each one memorized.

Keep the size of the difference in proportion. On 84 test penguins, the gap between these two test scores is **one penguin**. The penguins are a clean, easy dataset, and scaled k-NN in Section 3.3 did as well as either. A forest's advantage tends to be larger on messier data with more features; Module 11 shows a case where that happens.

### 5.3 Looking inside the forest

The 200 trees are stored in `rf.estimators_`, so we can ask each one for its opinion. One technical wrinkle: the trees inside a forest report classes as **numbers** (0, 1, 2), so we translate them back to names with `rf.classes_`.

In [ ]:
# RUN-TOGETHER
# Each tree's prediction for every test penguin: a 200 x 84 table of species names.
tree_votes = np.array([rf.classes_[t.predict(X_test.to_numpy()).astype(int)] for t in rf.estimators_])

# 1. How good is each tree on its own?
single_acc = (tree_votes == y_test.to_numpy()).mean(axis=1)
print(f'single trees, test accuracy:  min {single_acc.min():.3f}   mean {single_acc.mean():.3f}   max {single_acc.max():.3f}')
print(f'the forest, test accuracy  :  {rf.score(X_test, y_test):.3f}')

# 2. Which feature did each tree split on first?
first_split = pd.Series([X.columns[t.tree_.feature[0]] for t in rf.estimators_]).value_counts()
print()
print("each tree's first split:")
print(first_split)

# 3. Does a plain majority vote match the forest's prediction?
majority = pd.DataFrame(tree_votes).mode(axis=0).iloc[0].values
print()
print('majority vote agrees with rf.predict on', (majority == pred_rf).sum(), 'of', len(pred_rf), 'test penguins')

Three things to read off that output.

1. **The trees are individually mediocre and varied.** Their test accuracies span a wide range, and the forest beats their average.
2. **The trees really are different.** A single tree on all four features always opens with the same split. Because each split here may only consider a random 2 of the 4 features, the forest's trees open with all four of them.
3. **Voting and averaging agree.** On this data the majority vote and scikit-learn's averaging give identical predictions, as the implementation note in 5.1 said they should.

> **Discuss.** At least one of the 200 trees scored higher on the test set than the forest did. Why not throw the forest away and keep that one tree? Answer using the rule from Section 2 about the test set: could you have identified that tree before looking at the test scores?

---
## 6. Evaluating a Classifier
---
### 6.1 Accuracy, and why it lies

Suppose a bank builds a model to flag fraudulent phone calls. Out of 1,000 calls, 50 are fraud and 950 are legitimate. The model reports **97% accuracy**.

Before reading on: is that good?

Here is the model to beat. A "model" that ignores every feature and always predicts *legitimate* gets 950 of 1,000 right, which is **95% accuracy**, and it catches no fraud at all.

**Never report an accuracy without a baseline.** The right question is never "is 97% high?" but "what would nothing at all have scored?"

### 6.2 The confusion matrix

To see what a classifier is actually doing, break its predictions into four cells.

| | predicted fraud | predicted legitimate |
| --- | --- | --- |
| **actually fraud** | True Positive (TP) | False Negative (FN) |
| **actually legitimate** | False Positive (FP) | True Negative (TN) |

From those four numbers:

$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \qquad \text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN}$$

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

In words, and this is the part worth memorizing:

- **Precision:** of everything we flagged, how much was really fraud? (Low precision means crying wolf.)
- **Recall:** of all the real fraud, how much did we catch? (Low recall means missing it.)
- **F1:** a single number balancing the two, useful when classes are imbalanced.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/10-supervised_classification/fig_confusion_anatomy.png?raw=true" width="700">

*The same four numbers you are about to work with, laid out with the two metrics that come from them. Notice that precision reads down a column and recall reads across a row.*

#### **Activity 10.3 - Reading a confusion matrix by hand**

The bank's fraud model produced these results on 1,000 calls:

| | predicted fraud | predicted legitimate |
| --- | --- | --- |
| **actually fraud** | 30 | 20 |
| **actually legitimate** | 10 | 940 |

**Step 1. Identify the four cells.**

TP = ______   FN = ______   FP = ______   TN = ______

**Step 2. Compute, to three decimals.**

- accuracy = ______
- precision = ______
- recall = ______
- $F_1$ = ______

**Step 3. The baseline.** For the model that always predicts *legitimate*:

- accuracy = ______
- recall = ______

**Questions.**

**A.** Compare the model's accuracy to the baseline's. How much did all that modeling buy?

**B.** The model's recall and precision are quite different from each other. Which one is worse, and state in plain English what that failure means for the bank's customers.

**C.** The bank can lower the threshold so the model flags more calls. Predict what happens to precision and to recall, and explain why they move in opposite directions.

**D.** For fraud detection, which error is more costly: a false positive (a legitimate call flagged) or a false negative (fraud missed)? Now answer the same question for a spam filter, and for a cancer screening test. Does one answer serve all three?

#### **Class Example 10.3 - Confusion matrices in code**

In [ ]:
# RUN-TOGETHER
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, pred_tree, labels=tree.classes_)

sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=tree.classes_, yticklabels=tree.classes_)
plt.xlabel('predicted'); plt.ylabel('actual')
plt.title('Decision tree on the penguin test set')
plt.show()

print(classification_report(y_test, pred_tree))

`classification_report` gives precision, recall, and F1 **per class**, which matters: a model can be excellent on the common class and poor on a rare one, and a single overall accuracy hides that completely. The `support` column reports how many test cases each class actually had, and small support means the numbers beside it are jumpy.

> **Discuss.** Compare the per-class recall for the three species. Is the model equally good at all three? Which species would you want more training data for, and how did you decide?

#### **Spam Filter Example, Part 2 - Which tree is better?**

Part 1, in Section 4, grew two trees from the same eight emails:

- **Tree A** (one question): spam if links > 5, otherwise not spam.
- **Tree B** (up to two questions): spam if links > 5, or if links $\leq$ 5 **and** exclamation marks > 7; otherwise not spam.

On the eight emails they were grown from, Tree A gets 7 right (it misses email 7) and Tree B gets all 8. Section 2 says that comparison means nothing. The honest comparison uses emails **neither tree has seen**. Here is a test set of ten new emails, again invented for this example. Spam is the **positive** class, the thing we are trying to catch.

| email | exclamation marks | links | actual | Tree A predicts | Tree B predicts |
| --- | --- | --- | --- | --- | --- |
| T1 | 1 | 8 | spam | spam (**TP**) | spam (**TP**) |
| T2 | 9 | 6 | spam | spam (**TP**) | spam (**TP**) |
| T3 | 9 | 2 | spam | not spam (**FN**) | spam (**TP**) |
| T4 | 2 | 1 | spam | not spam (**FN**) | not spam (**FN**) |
| T5 | 1 | 0 | not spam | not spam (**TN**) | not spam (**TN**) |
| T6 | 4 | 3 | not spam | not spam (**TN**) | not spam (**TN**) |
| T7 | 8 | 2 | not spam | not spam (**TN**) | spam (**FP**) |
| T8 | 0 | 7 | not spam | spam (**FP**) | spam (**FP**) |
| T9 | 3 | 1 | not spam | not spam (**TN**) | not spam (**TN**) |
| T10 | 0 | 4 | not spam | not spam (**TN**) | not spam (**TN**) |

To fill in a prediction, run the email down the tree. For T3: links = 2 is at most 5, so Tree A stops and says *not spam*; Tree B asks a second question, and 9 exclamation marks is more than 7, so it says *spam*.

**Step 1. Count the four cells for each tree.**

*Tree A*

| | predicted spam | predicted not spam |
| --- | --- | --- |
| **actually spam** | TP = 2 (T1, T2) | FN = 2 (T3, T4) |
| **actually not spam** | FP = 1 (T8) | TN = 5 |

*Tree B*

| | predicted spam | predicted not spam |
| --- | --- | --- |
| **actually spam** | TP = 3 (T1, T2, T3) | FN = 1 (T4) |
| **actually not spam** | FP = 2 (T7, T8) | TN = 4 |

**Step 2. Compute the metrics.**

| | Tree A | Tree B | baseline: always "not spam" |
| --- | --- | --- | --- |
| accuracy | (2 + 5)/10 = **0.700** | (3 + 4)/10 = **0.700** | 6/10 = 0.600 |
| precision | 2/(2 + 1) = 0.667 | 3/(3 + 2) = 0.600 | not defined (it flags nothing) |
| recall | 2/(2 + 2) = 0.500 | 3/(3 + 1) = 0.750 | 0/4 = 0.000 |
| $F_1$ | $2 \cdot \frac{0.667 \cdot 0.500}{0.667 + 0.500}$ = 0.571 | $2 \cdot \frac{0.600 \cdot 0.750}{0.600 + 0.750}$ = 0.667 | not defined |

**Step 3. Read the result.**

- **Accuracy calls it a tie. The confusion matrices do not.** Tree B's second question caught one more spam (T3) and wrongly flagged one more real email (T7, perhaps an excited birthday message). One mistake was traded for another, so the accuracy did not move.
- **The trees fail in opposite directions.** Tree A is more trustworthy when it says "spam" (higher precision). Tree B lets less spam through (higher recall).
- **Neither tree catches T4**, a short spam message with one link and two exclamation marks. Nothing in the training emails looked like it, so no split was ever drawn around it.
- **Both trees flag T8**, the link-heavy email with no exclamation marks, because the first question sends it straight to a spam leaf. It is an email much like N2 from Part 1, and here it shows up as a real false positive.
- **Ten test emails is a very small test set.** Moving a single email changes each metric by 0.1 or more, so treat these numbers as an illustration of the reasoning, not as evidence about spam filters.

#### **Checking Part 2 in code**

In [ ]:
# RUN-TOGETHER
# The ten test emails, run through the two trees fitted in Part 1.
test_emails = pd.DataFrame({
    'exclamation marks': [1, 9, 9, 2, 1, 4, 8, 0, 3, 0],
    'links':             [8, 6, 2, 1, 0, 3, 2, 7, 1, 4],
    'label': ['spam'] * 4 + ['not spam'] * 6})
T = test_emails[['exclamation marks', 'links']]

order = ['spam', 'not spam']     # spam first, to match the tables above

for name, model in [('Tree A', spam_tree_a), ('Tree B', spam_tree_b)]:
    predicted = model.predict(T)
    print(f'=== {name} ===')
    print(pd.DataFrame(confusion_matrix(test_emails['label'], predicted, labels=order),
                       index=['actually spam', 'actually not spam'],
                       columns=['predicted spam', 'predicted not spam']))
    print()
    print(classification_report(test_emails['label'], predicted, labels=order))

Read the `spam` row of each report: its precision, recall, and F1 match the hand calculation. (The `not spam` row treats *not spam* as the positive class instead, which is a different question.)

> **Discuss.** You must deploy one of these two trees. Which would you choose for a filter that **deletes** suspected spam without showing it to anyone? Which for a filter that only adds a "possible spam" tag to the subject line? Use Activity 10.3 D: which mistake is more costly in each case?

---
## 7. Choosing a Classifier
---

| | k-nearest-neighbors | Decision tree | Random forest |
| --- | --- | --- | --- |
| Needs feature scaling | **Yes**, critically | No | No |
| Interpretable | Not really | **Yes**, you can read it | Not as a single rule |
| Main hyperparameters | `k` | `max_depth` | `n_estimators`, `max_features` |
| Training cost | none (stores data) | moderate | high (many trees) |
| Prediction cost | high (compares to everything) | very low | moderate (every tree votes) |
| Typical failure | wrong `k`, unscaled features | overfitting, instability | lost readability; little gain when one tree is already enough |

None of the three is "better". They fail differently, and knowing how a method fails is more useful than knowing its accuracy on someone else's dataset. The forest buys stability and often accuracy, and pays with the readable rule that made a single tree attractive; Module 11 shows how forests give back some interpretability through **feature importance**.

**Note:** regardless of algorithm:

1. Split before you model, and never let the test set influence training.
2. Every hyperparameter is a decision you must justify.
3. Report accuracy against a baseline, or do not report it.
4. A gap between training and test score is the signature of overfitting.

---
## Adding to the Toolkit
---

Every module from here to the final adds rows to one running table. By the last week you will have the whole collection, and the review will be about **choosing** from it: given the feature types you have and the question you were asked, which tool is right?

The columns never change:

| Tool | Question it answers | Feature types it needs | Label needed? | Key assumption | How it fails |
| --- | --- | --- | --- | --- | --- |
| **Train/test split** | Will this model work on data it has not seen? | any | **yes** | the test set stays untouched until the end | leaking the test set into any choice makes the estimate optimistic |
| **k-NN classifier** | Which category does this new case belong to? | interval or ratio, **scaled** | **yes**, categorical | nearby cases share labels | `k` changes the answer; unscaled features silently pick a winner; slow to predict |
| **Decision tree** | Which category, and by what readable rule? | any; **no scaling needed** | **yes**, categorical | axis-aligned cuts are enough | overfits without a depth limit; small data changes redraw the tree |
| **Random forest classifier** | Which category, as reliably as possible? | any; **no scaling needed** | **yes**, categorical | trees that make different mistakes cancel out when they vote | no single readable rule; vote share is not a calibrated probability; still answers confidently far from the data |
| **Confusion matrix + precision / recall** | What *kind* of mistakes is the classifier making? | predictions plus true labels | **yes** | the classes are meaningful | accuracy alone hides everything on imbalanced data |

**Compare this table with Module 9's.** The "Label needed?" column has flipped from no to yes, and that single change is the difference between the two halves of machine learning. Note also that the decision tree, and the random forest built from it, are the only tools so far that do **not** care about scaling, because a threshold on grams is unaffected by another column being in millimeters.

## Suggested Exercises

1. Five labeled training points, one feature: $x = 1$ (**A**), $x = 2$ (**A**), $x = 5$ (**B**), $x = 6$ (**B**), $x = 7$ (**B**). A new point arrives at $x = 4$.

    a. Compute the distance from the new point to each training point.

    b. Predict its label at $k = 1$, $k = 3$, and $k = 5$.

    c. At $k = 5$ every training point votes. Explain why that prediction would be the same no matter where the new point sat.

    d. A sixth training point is added at $x = 100$ with label **A**. Which of your three predictions change? What does that say about k-NN and outliers?

2. A node contains 12 cases: 8 of class X and 4 of class Y.

    a. A candidate split sends {7 X, 1 Y} left and {1 X, 3 Y} right. Compute the parent's Gini, both child Ginis, the weighted average, and the Gini gain.

    b. A second candidate sends {4 X, 2 Y} left and {4 X, 2 Y} right. Compute its gain without doing much arithmetic, and explain how you knew.

3. A medical test for a rare disease is evaluated on 10,000 patients. 100 have the disease. The test produces 80 true positives, 20 false negatives, 400 false positives, and 9,500 true negatives.

    a. Build the confusion matrix and compute accuracy, precision, recall, and F1.

    b. What accuracy would "always predict healthy" achieve?

    c. The test's precision is low. Explain, in terms a patient would understand, what that means for someone who receives a positive result.

    d. The manufacturer advertises "95% accurate." Is that claim false? Is it honest? Distinguish the two.

4. You train a decision tree with no depth limit and get 100% training accuracy and 71% test accuracy. A classmate proposes fixing this by collecting the test set into the training set, "so the model sees more data."

    a. Name the phenomenon the two scores reveal.

    b. Explain what would happen to the two reported numbers under your classmate's proposal, and why the result would be worse than useless.

    c. Give two changes that would genuinely help.

    d. You now try `max_depth` of 1 through 10 and report the best test accuracy. Explain the subtler problem with this procedure, using the rule from Section 2.

5. A spam filter is a random forest of 7 trees. On three incoming emails the trees vote:

    | email | spam votes | not-spam votes |
    | --- | --- | --- |
    | E1 | 7 | 0 |
    | E2 | 4 | 3 |
    | E3 | 1 | 6 |

    a. Give the forest's prediction for each email, and the winning vote share.

    b. A colleague reports E2 as "a 57% chance of spam." Explain why that sentence claims more than the vote shows.

    c. The forest was trained on 1,000 emails. Roughly how many of them would you expect to be left out of any one tree's bootstrap sample? Show the calculation.

    d. A classmate adds 7 more trees, all exact copies of the first tree. Will the forest improve? Explain using the idea from Section 5.1 that the trees must make different mistakes.
